# Testing speed of data import

## Loading packages

In [1]:
using Revise
using Pkg;
#Pkg.develop(path="/home/gert/Projects/FusionRings.jl/")
#Pkg.develop(path="/Users/gertvercleyen/Projects/FusionRings.jl/")
using FusionRings
using Oscar
using JSON
using Base.Threads
using LinearAlgebra

[ Info: Precompiling FusionRings [609e252f-52eb-439d-abe9-2fbe746db898] (cache misses: incompatible header (4))

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up


ERROR: Method overwriting is not permitted during Module precompilation. Use `__precompile__(false)` to opt-out of precompilation.
┌ Info: Skipping precompilation due to precompilable error. Importing FusionRings [609e252f-52eb-439d-abe9-2fbe746db898].
└   exception = Error when precompiling module, potentially caused by a __precompile__(false) declaration in the module.



Welcome to Nemo version 0.52.4



SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up



Nemo comes with absolutely no warranty whatsoever
 ┌───────┐   GAP 4.15.1 of 2025-10-18
 │  GAP  │   https://www.gap-system.org
 └───────┘   Architecture: x86_64-pc-linux-gnu-julia1.12-64-kv10
 Configuration:  gmp 6.3.0, Julia GC, Julia 1.12.3, readline
 Loading the library and packages ...
 Packages:   AClib 1.3.3, Alnuth 3.2.1, AtlasRep 2.1.9, AutoDoc 2025.10.16, 
             AutPGrp 1.11.1, Browse 1.8.21, CaratInterface 2.3.7, CRISP 1.4.8, 
             Cryst 4.1.30, CrystCat 1.1.10, CTblLib 1.3.11, 
             curlInterface 2.4.2, FactInt 1.6.3, FGA 1.5.0, Forms 1.2.13, 
             GAPDoc 1.6.7, genss 1.6.9, IO 4.9.3, IRREDSOL 1.4.4, 
             JuliaInterface 0.16.2, LAGUNA 3.9.7, orb 5.0.1, 
             PackageManager 1.6.3, Polenta 1.3.11, Polycyclic 2.17, 
             PrimGrp 4.0.1, RadiRoot 2.9, recog 1.4.4, ResClasses 4.7.4, 
             SmallGrp 1.5.4, Sophus 1.27, SpinSym 1.5.2, StandardFF 1.0, 
             TomLib 1.2.11, TransGrp 3.6.5, utils 0.92
 Try '??help'

## parallel_load

The data we will test on are
* The unique fpdims of all fusion rings, stored in "~/Tests/fpdims"
* The unique values of all characters of all fusion rings, stored in "~/tests/characters"

Each mrdi file contains a single qqbar number

In [7]:
base_dir = "/home/gert/Tests";
load_fp_dim(i::Int64) = Oscar.load( base_dir * "/fpdims/fpdim_" * string(i) * ".mrdi" );
ndims = 50903;

50903

In [5]:
function parallel_load( ) 

FR(2, 1, 0, 2)

In [4]:
dims = unique( vcat( [ fpdims(r) for r in frl ] ... ) );

In [8]:
@time [ load_fp_dim(i) for i in 1:ndims ]

 20.384064 seconds (37.39 M allocations: 2.403 GiB, 4.93% gc time, 18.59% compilation time)


50903-element Vector{QQBarFieldElem}:
 {a1: 1.00000}
 {a2: 1.61803}
 {a2: 1.41421}
 {a1: 2.00000}
 {a3: 1.80194}
 {a3: 2.24698}
 {a2: 2.41421}
 {a2: 2.61803}
 {a3: 1.87939}
 {a3: 2.53209}
 {a3: 2.87939}
 {a2: 1.73205}
 {a2: 2.30278}
 ⋮
 {a3: 17.5456}
 {a3: 24.1246}
 {a3: 8.29757}
 {a3: 19.1841}
 {a3: 17.4087}
 {a3: 21.7427}
 {a3: 16.6708}
 {a3: 24.8531}
 {a3: 6.34780}
 {a3: 18.2512}
 {a3: 9.46544}
 {a3: 20.3365}

In [12]:
@time begin 
    l = [ [], [], [], [] ]
    @threads for i in 1:length(dims)
        push!( l[Threads.threadid()-1], load_fp_dim(i) )
    end
    vcat( l... )
end

  5.021746 seconds (30.80 M allocations: 2.086 GiB, 16.49% gc time, 287 lock conflicts, 1.97% compilation time)


50903-element Vector{Any}:
 {a1: 1.00000}
 {a2: 1.61803}
 {a2: 1.41421}
 {a1: 2.00000}
 {a3: 1.80194}
 {a3: 2.24698}
 {a2: 2.41421}
 {a2: 2.61803}
 {a3: 1.87939}
 {a3: 2.53209}
 {a3: 2.87939}
 {a2: 1.73205}
 {a2: 2.30278}
 ⋮
 {a4: 28.5713}
 {a4: 9.91651}
 {a4: 23.8773}
 {a4: 24.5608}
 {a4: 14.7502}
 {a4: 19.7831}
 {a4: 20.5152}
 {a2: 33.6828}
 {a2: 34.6828}
 {a4: 17.5212}
 {a4: 22.5683}
 {a4: 25.3004}

In [18]:
function parallel_load( dirname::String )
    filenames = readdir( dirname )
    l = fill( [], Threads.nthreads() )
    @threads for fn in filenames
        push!( l[Threads.threadid()-1], Oscar.load(dirname * fn) )
    end
    vcat( l... )
end

parallel_load (generic function with 1 method)

In [19]:
@time chars = parallel_load( "/home/gert/Tests/characters/" );

 43.009472 seconds (237.02 M allocations: 15.184 GiB, 13.23% gc time, 6085 lock conflicts, 14.57% compilation time)


In [26]:
flat_chars = vcat( [ vcat( ch...) for ch in chars ]...)

3335952-element Vector{QQBarFieldElem}:
 {a1: 1.00000}
 {a1: 1.00000}
 {a1: 1.00000}
 {a2: 10.0990}
 {a2: -0.0990195}
 {a1: 1.00000}
 {a1: 1.00000}
 {a2: 11.0902}
 {a2: -0.0901699}
 {a1: 1.00000}
 {a1: 1.00000}
 {a2: 12.0828}
 {a2: -0.0827625}
 ⋮
 {a2: -0.500000 - 0.866025*im}
 {a2: -0.500000 + 0.866025*im}
 {a1: 1.00000}
 {a1: 1.00000}
 {a1: 1.00000}
 {a2: -0.500000 + 0.866025*im}
 {a2: -0.500000 - 0.866025*im}
 {a2: -0.500000 - 0.866025*im}
 {a2: -0.500000 + 0.866025*im}
 {a2: -0.500000 + 0.866025*im}
 {a2: -0.500000 - 0.866025*im}
 {a1: 1.00000}

In [27]:
unique_flat_chars = unique( flat_chars );

In [37]:
allnumbers = union( unique_flat_chars, vcat( l... ) );

In [42]:
unique(typeof.(allnumbers))

1-element Vector{DataType}:
 QQBarFieldElem

In [44]:
unique(degree.(minimal_polynomial.( allnumbers )))

9-element Vector{Int64}:
 1
 2
 3
 5
 6
 4
 7
 8
 9

In [45]:
length(allnumbers)

220027

In [51]:
replace( string(minpoly(allnumbers[234])),  "*" => "", " " => ""  )

"x^3-20x^2-26x+8"

In [105]:
function qqb_id( x::QQBarFieldElem ) 
    mp = minimal_polynomial(x)
    degstring = string( degree( mp ) )
    polstring = 
        replace( 
            string(mp),  
            "*" => "", " " => ""  
        )
    numstring = string( rootnum( x ) )

    degstring * "_" * polstring * "_" * numstring
    
end

function rootnum( x::QQBarFieldElem )
    p   = minimal_polynomial( x ) 
    rts = roots( QQBar, p )
    sr  = sort( rts, by = root_sort_crit )
    findfirst( y -> y == x, sr )
end

function root_sort_crit( x )
    ( - Int( is_real( x ) ), real(x), imag(x) )
end

function is_saved( dict, x::QQBarFieldElement )
    x ∈ values( dict )
end

function save_qqb_nums( id_file_name, num_file_name, v::Vector{QQBarFieldElem} )
    idstrings = qqb_id.(x)
    open( id_file_name, "a" ) do io
        for s in idstrings 
            write(io, s)
        end
    end
    num_data = Oscar.load(num_file_name)
    append!( num_data, v ) 
    Oscar.save( num_file_name, x )
end

function save_qqb_num( x::QQBarFieldElem )
    save_qqb_num( joinpath(@__DIR__, "data","QQBarFieldElems"), x )
end

save_qqb_num (generic function with 2 methods)

In [107]:
@threads for n in allnumbers 
    save_qqb_num( "/home/gert/Projects/FusionRings.jl/src/data/Numbers/QQBarFieldElems/", n )
end

In [117]:
Oscar.save( "/home/gert/Projects/FusionRings.jl/src/data/Numbers/QQBarFieldElems/qqbfieldelems.mrdi", allnumbers )

In [119]:
Oscar.save( "/home/gert/Projects/FusionRings.jl/src/data/Numbers/QQBarFieldElems/idsqqbfieldelems.mrdi", qqb_id.(allnumbers) )

In [120]:
function load_qqb_num_dict()
    ids  = Oscar.load("/home/gert/Projects/FusionRings.jl/src/data/Numbers/QQBarFieldElems/idsqqbfieldelems.mrdi")
    nums = Oscar.load("/home/gert/Projects/FusionRings.jl/src/data/Numbers/QQBarFieldElems/qqbfieldelems.mrdi")
    Dict( ids[i] => nums[i] for i in 1:length(ids) )
end

load_qqb_num_dict (generic function with 1 method)

In [123]:
@time d = load_qqb_num_dict();

 63.231371 seconds (106.75 M allocations: 7.338 GiB, 3.92% gc time)


In [125]:
d["1_x-1_1"]

{a1: 1.00000}

In [ ]:

JSON.print(io::IO, a::AbstractDict, indent)

In [129]:
?ismissing

search: ismissing missing skipmissing Missing isdisjoint isnothing SeriesRing



```julia
ismissing(x)
```

Indicate whether `x` is [`missing`](@ref).

See also: [`skipmissing`](@ref), [`isnothing`](@ref), [`isnan`](@ref).


In [137]:
function mtfromjs( js )
  jsmt = js["mult_tab"]
  r = length(jsmt)
  mt = zeros(Int, r, r, r)
  for i in 1:r, j in 1:r, k in 1:r 
      mt[i,j,k] = Int.(jsmt[i][j][k])
  end
  mt
end

function fcfromjs( js::JSON.Object{String, Any} )
  fc = js["formal_code"]
  if length(fc) == 0
    missing
  else
    [ fc[i] for i in 1:4 ]  
  end
end

fcfromjs (generic function with 1 method)

In [154]:
jst = JSON.parsefile( "/home/gert/Projects/FusionRings.jl/src/data/FusionRingsJSON/ring8.json" ); 

In [167]:
jst["sub_fusion_rings"]["value"][1]

2-element Vector{Any}:
 Any[1, 2]
 Any[2, 1, 0, 1]

In [163]:
function tpdfromjs(js::JSON.Object{String, Any})
  tps = js["tensor_product_decompositions"]["value"]
  [ [ Int.( decomp[i] ) for i in 1:length(decomp) ] for decomp in tps ]
end

tpdfromjs (generic function with 1 method)

In [168]:
function sfrfromjs(js::JSON.Object{String, Any})
  srs = js["sub_fusion_rings"]["value"]
  [ [ Int.( sr[i] ) for i in 1:length(sr) ] for sr in srs ]
end

sfrfromjs (generic function with 1 method)

In [169]:
sfrfromjs( jst )

3-element Vector{Vector{Vector{Int64}}}:
 [[1, 2], [2, 1, 0, 1]]
 [[1, 3], [2, 1, 0, 1]]
 [[1, 4], [2, 1, 0, 1]]

In [170]:
function sfrfromjs(js::JSON.Object{String, Any})
  srs = js["sub_fusion_rings"]["value"]
  intData =  [ [ Int.( sr[i] ) for i in 1:length(sr) ] for sr in srs ]
  [ Dict( "injection" => data[1] , "anyonwiki_code" => data[2] ) for data in intData]
end

sfrfromjs2 (generic function with 1 method)

In [183]:
function vec_to_cflt( v::Vector{Any} )::ComplexF64
    v[1] + v[2]*1im
end

function ncfromjs(js::JSON.Object{String, Any})::Matrix{ComplexF64}
    ncvecs = jst["numeric_characters"]
    r = length(ncvecs)
    [ vec_to_cflt( ncvecs[i][j] ) for i in 1:r, j in 1:r ]
end

ncfromjs (generic function with 1 method)

In [189]:
function nfpdsfromjs(js::JSON.Object{String, Any})::Vector{ComplexF64}
    nfpdims = js["numeric_frobenius_perron_dimensions"]
    vec_to_cflt.( nfpdims )
end

nfpdsfromjs (generic function with 1 method)

In [191]:
jst["categorifiable"]

LoadError: KeyError: key "categorifiable" not found